# IMPORT LIBRARIES

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.cluster.hierarchy import dendrogram, linkage
import matplotlib.pyplot as plt

# LOAD DATASETS

In [2]:
deliveries = pd.read_csv("../data/raw_data/ball_by_ball_data.csv", encoding="latin1")

In [3]:
deliveries.head()

,season_id,match_id,batter,bowler,non_striker,team_batting,team_bowling,over_number,ball_number,batter_runs,...,is_bye,is_penalty,wide_ball_runs,no_ball_runs,leg_bye_runs,bye_runs,penalty_runs,wicket_kind,is_super_over,innings
0,2008,335982,SC Ganguly,P Kumar,BB McCullum,Kolkata Knight Riders,Royal Challengers Bangalore,0,0,0,...,False,False,0,0,1,0,0,NaN,False,1
1,2008,335982,BB McCullum,P Kumar,SC Ganguly,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,0,...,False,False,0,0,0,0,0,NaN,False,1
2,2008,335982,BB McCullum,P Kumar,SC Ganguly,Kolkata Knight Riders,Royal Challengers Bangalore,0,2,0,...,False,False,1,0,0,0,0,NaN,False,1
3,2008,335982,BB McCullum,P Kumar,SC Ganguly,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,0,...,False,False,0,0,0,0,0,NaN,False,1
4,2008,335982,BB McCullum,P Kumar,SC Ganguly,Kolkata Knight Riders,Royal Challengers Bangalore,0,4,0,...,False,False,0,0,0,0,0,NaN,False,1


In [4]:
deliveries.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 278205 entries, 0 to 278204
Data columns (total 30 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   season_id          278205 non-null  int64 
 1   match_id           278205 non-null  int64 
 2   batter             278205 non-null  object
 3   bowler             278205 non-null  object
 4   non_striker        278205 non-null  object
 5   team_batting       278205 non-null  object
 6   team_bowling       278205 non-null  object
 7   over_number        278205 non-null  int64 
 8   ball_number        278205 non-null  int64 
 9   batter_runs        278205 non-null  int64 
 10  extras             278205 non-null  int64 
 11  total_runs         278205 non-null  int64 
 12  batsman_type       278205 non-null  object
 13  bowler_type        278205 non-null  object
 14  player_out         13823 non-null   object
 15  fielders_involved  13823 non-null   object
 16  is_wicket          2

In [5]:
deliveries.columns

Index(['season_id', 'match_id', 'batter', 'bowler', 'non_striker',
       'team_batting', 'team_bowling', 'over_number', 'ball_number',
       'batter_runs', 'extras', 'total_runs', 'batsman_type', 'bowler_type',
       'player_out', 'fielders_involved', 'is_wicket', 'is_wide_ball',
       'is_no_ball', 'is_leg_bye', 'is_bye', 'is_penalty', 'wide_ball_runs',
       'no_ball_runs', 'leg_bye_runs', 'bye_runs', 'penalty_runs',
       'wicket_kind', 'is_super_over', 'innings'],
      dtype='object')

In [6]:
deliveries.shape

(278205, 30)

- <b>Total runs of each batsmans.

In [7]:
runs = deliveries.groupby('batter')['batter_runs'].sum().reset_index()

runs.rename(columns={'batter_runs': 'batsman_total_runs'}, inplace=True)

In [8]:
runs.head()

,batter,batsman_total_runs
0,A Ashish Reddy,280
1,A Badoni,963
2,A Chandila,4
3,A Chopra,53
4,A Choudhary,25


- <b> Total balls faced by each batsman.

In [9]:
balls = deliveries.groupby('batter')['ball_number'].count().reset_index()

balls.rename(columns={'ball_number': 'balls_faced'}, inplace=True)

In [10]:
balls.head()

,batter,balls_faced
0,A Ashish Reddy,196
1,A Badoni,740
2,A Chandila,7
3,A Chopra,75
4,A Choudhary,20


- <b> Strike rate of each batsman.

In [11]:
batsman_df = runs.merge(balls, on='batter')

In [12]:
batsman_df.head()

,batter,batsman_total_runs,balls_faced
0,A Ashish Reddy,280,196
1,A Badoni,963,740
2,A Chandila,4,7
3,A Chopra,53,75
4,A Choudhary,25,20


In [13]:
batsman_df['strike_rate'] = (batsman_df['batsman_total_runs']*100)/ batsman_df['balls_faced']

In [14]:
batsman_df.head()

,batter,batsman_total_runs,balls_faced,strike_rate
0,A Ashish Reddy,280,196,142.857143
1,A Badoni,963,740,130.135135
2,A Chandila,4,7,57.142857
3,A Chopra,53,75,70.666667
4,A Choudhary,25,20,125.000000


- <b> Total 4's by each batsman.

In [15]:
fours = deliveries[deliveries['batter_runs'] == 4].groupby('batter').size().reset_index(name='fours')

- <b> Total 6's by each batsman.

In [16]:
sixes = deliveries[deliveries['batter_runs'] == 6].groupby('batter').size().reset_index(name='sixes')

In [17]:
print(fours.head())
print('_'*50)
print('_'*50)
print(sixes.head())

           batter  fours
0  A Ashish Reddy     16
1        A Badoni     73
2        A Chopra      7
3     A Choudhary      1
4      A Flintoff      5
__________________________________________________
__________________________________________________
           batter  sixes
0  A Ashish Reddy     15
1        A Badoni     38
2     A Choudhary      1
3      A Flintoff      2
4       A Manohar     14


In [18]:
batsman_df = batsman_df.merge(fours, on='batter', how='left')
batsman_df = batsman_df.merge(sixes, on='batter', how='left')

In [19]:
batsman_df.head()

,batter,batsman_total_runs,balls_faced,strike_rate,fours,sixes
0,A Ashish Reddy,280,196,142.857143,16.0,15.0
1,A Badoni,963,740,130.135135,73.0,38.0
2,A Chandila,4,7,57.142857,NaN,NaN
3,A Chopra,53,75,70.666667,7.0,NaN
4,A Choudhary,25,20,125.000000,1.0,1.0


In [20]:
batsman_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 703 entries, 0 to 702
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   batter              703 non-null    object 
 1   batsman_total_runs  703 non-null    int64  
 2   balls_faced         703 non-null    int64  
 3   strike_rate         703 non-null    float64
 4   fours               559 non-null    float64
 5   sixes               469 non-null    float64
dtypes: float64(3), int64(2), object(1)
memory usage: 33.1+ KB


In [21]:
batsman_df.fillna(0, inplace=True)

- <b> Boundary percentage.

In [22]:
batsman_df['boundary_percentage'] = ((batsman_df['fours'] * 4 + batsman_df['sixes'] * 6)/batsman_df['batsman_total_runs'])*100

In [23]:
batsman_df.head()

,batter,batsman_total_runs,balls_faced,strike_rate,fours,sixes,boundary_percentage
0,A Ashish Reddy,280,196,142.857143,16.0,15.0,55.000000
1,A Badoni,963,740,130.135135,73.0,38.0,53.997923
2,A Chandila,4,7,57.142857,0.0,0.0,0.000000
3,A Chopra,53,75,70.666667,7.0,0.0,52.830189
4,A Choudhary,25,20,125.000000,1.0,1.0,40.000000


In [24]:
batsman_df.shape

(703, 7)

- <b> Remove low matches players.

In [25]:
batsman_df = batsman_df[batsman_df['balls_faced'] >=450]

In [26]:
batsman_df.shape

(142, 7)

- <b> Select Features

In [27]:
features = batsman_df[['batsman_total_runs','balls_faced','strike_rate','fours','sixes','boundary_percentage']]

- <b>Scale Features

In [28]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

scaled_features = scaler.fit_transform(features)

In [29]:
scaled_features

array([[-7.45252567e-01, -7.46919067e-01, -4.14664854e-02,
        -8.06090544e-01, -7.14753264e-01, -9.62130947e-01],
       [-7.38113966e-01, -7.12063642e-01, -4.46898526e-01,
        -7.99379424e-01, -6.69724442e-01, -7.26106715e-01],
       [ 1.99207639e+00,  1.58839440e+00,  1.33747594e+00,
         1.48240127e+00,  2.51231227e+00,  7.76993572e-02],
       [-2.74986950e-02, -5.40612310e-02,  1.76792831e-01,
         3.07955323e-01,  9.57655216e-02,  1.74144746e+00],
       [-9.00354895e-01, -8.66787724e-01, -7.34254609e-01,
        -1.00071301e+00, -8.49839728e-01, -1.77164754e+00],
       [ 3.52794044e-01,  6.29816334e-03,  2.43670103e+00,
        -4.10229006e-02,  2.06202406e+00,  2.55495835e+00],
       [-1.25725295e-02,  6.58074254e-02, -5.48783162e-01,
         1.40177331e-01, -1.14368978e-01,  3.67762190e-01],
       [-4.35696873e-01, -4.46822360e-01,  7.90702073e-02,
        -5.51067996e-01, -4.29570728e-01, -8.78334791e-01],
       [-9.34101009e-01, -8.76139179e-01, -1.226

- <b> K-Means Clustering

In [30]:
from sklearn.cluster import KMeans, AgglomerativeClustering

In [31]:
kmeans = KMeans(
    n_clusters=4,
    random_state=42
)

batsman_df['kmeans_cluster'] = kmeans.fit_predict(scaled_features)

In [32]:
hierarchical = AgglomerativeClustering(
    n_clusters=4
)

batsman_df['hierarchical_cluster'] = hierarchical.fit_predict(scaled_features)

- <b> Save Model

In [33]:
import joblib

In [34]:
joblib.dump(kmeans, "../models/kmeans_model.pkl")
joblib.dump(hierarchical, "../models/hierarchical_model.pkl")
joblib.dump(scaler, "../models/clustering_scaler.pkl")

['../models/clustering_scaler.pkl']

- <b> Cluster labelling

In [35]:
cluster_names = {
    0: "Anchor Batter",
    1: "Aggressive Finisher",
    2: "Power Hitter",
    3: "Balanced Batter"
}

batsman_df['batsman_type'] = batsman_df['kmeans_cluster'].map(cluster_names)

- <b> Save Player Data

In [36]:
batsman_df.to_csv("../models/batsman_clusters.csv", index=False)

In [37]:
batsman_clusters = pd.read_csv("../models/batsman_clusters.csv")

In [38]:
batsman_clusters

,batter,batsman_total_runs,balls_faced,strike_rate,fours,sixes,boundary_percentage,kmeans_cluster,hierarchical_cluster,batsman_type
0,A Badoni,963,740,130.135135,73.0,38.0,53.997923,0,2,Anchor Batter
1,A Symonds,974,781,124.711908,74.0,41.0,55.646817,0,2,Anchor Batter
2,AB de Villiers,5181,3487,148.580442,414.0,253.0,61.262305,2,1,Power Hitter
3,AC Gilchrist,2069,1555,133.054662,239.0,92.0,72.885452,1,0,Aggressive Finisher
4,AD Mathews,724,599,120.868114,44.0,29.0,48.342541,0,2,Anchor Batter
...,...,...,...,...,...,...,...,...,...,...
137,WP Saha,2934,2368,123.902027,296.0,87.0,58.145876,1,2,Aggressive Finisher
138,Y Venugopal Rao,985,865,113.872832,77.0,37.0,53.807107,0,2,Anchor Batter
139,YBK Jaiswal,2166,1454,148.968363,259.0,92.0,73.314866,3,0,Balanced Batter
140,YK Pathan,3222,2334,138.046272,263.0,161.0,62.631906,1,3,Aggressive Finisher
